In [1]:
# test with supervised optimal transport loss
# comparison with unsupervised

import random
import torch
import numpy as np
import torch.nn as nn
from tqdm import tqdm
import torch.nn.functional as F
from torch.amp import GradScaler
from torch.utils.data import DataLoader
import deeplake
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score
from sklearn.decomposition import PCA
from geomloss import SamplesLoss
import time
import pandas as pd
from torch.utils.data import ConcatDataset, RandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR
import pickle
import seaborn as sns
import itertools
from dataset_OT import make_multi_WSI_dataset


batch_size = 512 

print('Loading starting...')
idx_range_subset1 = [i for i in range(1,52+1)]
random.shuffle(idx_range_subset1)
num_train = int(np.ceil(0.7 * len(idx_range_subset1))) 
train_range1, val_range1 = idx_range_subset1[:num_train], idx_range_subset1[num_train:]

idx_range_subset3 = [i for i in range(1,26+1)] 
random.shuffle(idx_range_subset3)
num_train = int(np.ceil(0.7 * len(idx_range_subset3))) 
train_range3, val_range3 = idx_range_subset3[:num_train], idx_range_subset3[num_train:]

akoya_loader_train_subset1 = make_multi_WSI_dataset('Subset1', train_range1, ['Akoya'], train_or_test='Train', batch_size=batch_size)
akoya_loader_val_subset1 = make_multi_WSI_dataset('Subset1', val_range1, ['Akoya'], train_or_test='Train', batch_size=batch_size)
akoya_loader_train_subset3 = make_multi_WSI_dataset('Subset3', train_range3, ['Akoya'], train_or_test='Train', batch_size=batch_size)
akoya_loader_val_subset3 = make_multi_WSI_dataset('Subset3', val_range3, ['Akoya'], train_or_test='Train', batch_size=batch_size)
leica_loader_train = make_multi_WSI_dataset('Subset3', train_range3, ['Leica'], train_or_test='Train', batch_size=batch_size)
leica_loader_val = make_multi_WSI_dataset('Subset3', val_range3, ['Leica'], train_or_test='Train', batch_size=batch_size)

akoya_loader_train = ConcatDataset([akoya_loader_train_subset1, akoya_loader_train_subset3])
akoya_loader_val = ConcatDataset([akoya_loader_val_subset1, akoya_loader_val_subset3])

len_akoya_train = len(akoya_loader_train)
len_akoya_val = len(akoya_loader_val)

len_leica_train = len(leica_loader_train)
len_leica_val = len(leica_loader_val)

len_train = len_akoya_train + len_leica_train
len_val = len_akoya_val + len_leica_val

B_A_train = round(batch_size * len_akoya_train / (len_akoya_train + len_leica_train))
B_L_train = batch_size - B_A_train 
B_A_val = round(batch_size * len_akoya_val / (len_akoya_val + len_leica_val))
B_L_val = batch_size - B_A_val

akoya_loader_train = DataLoader(akoya_loader_train, batch_size=B_A_train, shuffle=True, num_workers=6, pin_memory=True, persistent_workers=True)
akoya_loader_val = DataLoader(akoya_loader_val, batch_size=B_A_val, num_workers=6, pin_memory=True, persistent_workers=True)
leica_loader_train = DataLoader(leica_loader_train, batch_size=B_L_train, shuffle=True, num_workers=6, pin_memory=True, persistent_workers=True)
leica_loader_val = DataLoader(leica_loader_val, batch_size=B_L_val, num_workers=6, pin_memory=True, persistent_workers=True)

#leica_train_iter = itertools.cycle(iter(leica_loader_train))
#leica_val_iter = itertools.cycle(iter(leica_loader_val)) #????

print("Train batches Akoya:", len(akoya_loader_train), 'batch size:', B_A_train)
print("Train batches Leica:", len(leica_loader_train), 'batch size:', B_L_train)
print("Validation batches Akoya:", len(akoya_loader_val), 'batch size:', B_A_val)
print("Validation batches Leica:", len(leica_loader_val), 'batch size:', B_L_val)

#in samples
print('len train:', len_train)


loss_geom = SamplesLoss('sinkhorn', p=2, blur=0.1, scaling=0.95, verbose=False)


Loading starting...
Train batches Akoya: 5034 batch size: 402
Train batches Leica: 5049 batch size: 110
Validation batches Akoya: 1208 batch size: 376
Validation batches Leica: 1207 batch size: 136
len train: 2578771


In [2]:
# unsupervised
loss_geom = SamplesLoss('sinkhorn', p=2, blur=0.1, scaling=0.95, verbose=False)

#supervised
def compute_supervised_ot_loss(embedding_akoya, embedding_leica, labels_akoya, labels_leica):
    """
    Fast supervised OT loss by applying OT only between embeddings with the same label.
    Skip labels that are missing from either scanner in the current batch.
    """
    # Fast intersection using torch operations
    akoya_labels_set = torch.unique(labels_akoya)
    leica_labels_set = torch.unique(labels_leica)
    
    # Use broadcasting to find intersection efficiently
    common_mask = akoya_labels_set.unsqueeze(1) == leica_labels_set.unsqueeze(0)
    if not common_mask.any():
        return torch.tensor(0.0, device=embedding_akoya.device, requires_grad=True)
    
    common_labels = akoya_labels_set[common_mask.any(dim=1)]
    
    # Pre-allocate lists for concatenation (faster than accumulating)
    ot_losses = []
    
    # Vectorized label processing
    for label in common_labels:
        # Boolean indexing (faster than where)
        akoya_idx = labels_akoya == label
        leica_idx = labels_leica == label
        
        akoya_emb = embedding_akoya[akoya_idx]
        leica_emb = embedding_leica[leica_idx]
        
        # Skip if either is empty (shouldn't happen with our intersection logic, but safety check)
        if akoya_emb.shape[0] == 0 or leica_emb.shape[0] == 0:
            continue
            
        # Ensure 2D shape: (N, D) - geomloss requirement
        if akoya_emb.dim() > 2:
            akoya_emb = akoya_emb.flatten(0, -2)
        if leica_emb.dim() > 2:
            leica_emb = leica_emb.flatten(0, -2)
            
        # Compute OT loss for this label
        ot_losses.append(loss_geom(akoya_emb, leica_emb))
    
    # Fast averaging using torch.stack
    if ot_losses:
        return torch.stack(ot_losses).mean()
    else:
        return torch.tensor(0.0, device=embedding_akoya.device, requires_grad=True)

In [3]:
# i have to try on the gpu!!

dic={0:[], 1:[],2:[],3:[],4:[]}
def supervised_OT_loss(embedding_akoya, embedding_leica, labels_akoya, labels_leica):
    OT_loss = 0.0
    device = embedding_akoya.device
    
    
    common_labels = np.intersect1d(torch.unique(labels_akoya), torch.unique(labels_leica))
    
    for label in common_labels:
        akoya_idx = labels_akoya == label
        leica_idx = labels_leica == label
        akoya_emb = embedding_akoya[akoya_idx]
        leica_emb = embedding_leica[leica_idx]
        
        '''if akoya_numel == 0 or leica_emb.numel() == 0: 
            continue'''
        #print(f'weight for label {label}:', 1/leica_idx.sum())
        label_OT = loss_geom(akoya_emb, leica_emb)
        OT_loss += label_OT  #* (1/leica_idx.sum())
        dic[label].append(label_OT)
        
    
    return OT_loss
#try the version with the modified cost matrix

In [4]:
go = False
if go:
    total_time = 0
    for batch_akoya, batch_leica in tqdm(zip(akoya_loader_train, leica_loader_train),
                                                    desc=f"Epoch 0 Training - OT in progress",
                                                    total=min(len(akoya_loader_train), len(leica_loader_train))):
                #for batch_akoya in tqdm(akoya_loader_train, desc=f'Epoch {epoch} Training - Optimal Transport in progress'):
                    #batch_leica = next(leica_train_iter)
        
        embedding_akoya = batch_akoya['embedding'] 
        embedding_leica = batch_leica['embedding'] 
        start = time.time()
        loss_geom(embedding_akoya, embedding_leica)
        end = time.time()
        total_time += end - start
    
    print('average time per batch', total_time/min(len(akoya_loader_train), len(leica_loader_train)))

In [5]:
test = True
lu_OT = []
ls_OT = []
if test:
    total_time = 0
    for batch_akoya, batch_leica in tqdm(zip(akoya_loader_train, leica_loader_train),
                                                    desc=f"Epoch 0 Training - OT in progress",
                                                    total=min(len(akoya_loader_train), len(leica_loader_train))):
                #for batch_akoya in tqdm(akoya_loader_train, desc=f'Epoch {epoch} Training - Optimal Transport in progress'):
                    #batch_leica = next(leica_train_iter)
        
        embedding_akoya = batch_akoya['embedding'] 
        embedding_leica = batch_leica['embedding'] 
        labels_akoya = batch_akoya['label']
        labels_leica = batch_leica['label']
        start = time.time()
        #s_OT = supervised_OT_loss(embedding_akoya, embedding_leica, labels_akoya, labels_leica)
        u_OT = loss_geom(embedding_akoya, embedding_leica)
        lu_OT.append(u_OT)
        #ls_OT.append(s_OT)
        end = time.time()
        total_time += end - start
        #print(end-start)



    

#print('average time per batch', total_time/min(len(akoya_loader_train), len(leica_loader_train)))

Epoch 0 Training - OT in progress: 100%|███████████████████████████████████████████| 5034/5034 [23:55<00:00,  3.51it/s]


In [ ]:
import matplotlib.pyplot as plt
lu_OT_array = np.array(lu_OT) #*0.1
ls_OT_array = np.array(ls_OT) #* 0.01


plt.figure(figsize=(10, 6))
#plt.plot(lu_OT_array, label='u_OT', color='blue')
#no weigthing scheme
#plt.plot(dic[0], label='s_OT') #std 16, median 509
#plt.plot(dic[1], label='s_OT') #std 30 median 440
#plt.plot(dic[2], label='s_OT') #std 12, median 420
#plt.plot(dic[3], label='s_OT') #std 10, median 433
#plt.plot(dic[4], label='s_OT') #std 70, median 590
#its as if the OT based loss was already performing the weighting scheme for  G5
#plt.plot(ls_OT_array, label='lu_OT') #std 302, median 2300
plt.plot(lu_OT_array, label='lu')

# Add labels and legend

plt.title('Plot of u_OT and s_OT')
plt.legend()
plt.grid(True)

# Show the plot
plt.show()
plt.savefig('label-OT.pdf')



# Basic statistics
print("Basic Statistics for lu_OT:")
print(f"Mean: {np.mean(lu_OT_array)}")
print(f"Median: {np.median(lu_OT_array)}")
print(f"Standard Deviation: {np.std(lu_OT_array)}")
print(f"Variance: {np.var(lu_OT_array)}")
print(f"Minimum: {np.min(lu_OT_array)}")
print(f"Maximum: {np.max(lu_OT_array)}")



#with the scaling factor 0.1, the std is below one for unsupervised learning, 
#even with the scaling factor, the std of my supervised technique is high (=22)
#lets understand why: too much weight given to G5
#analyse the distribution of label OT losses to understand how we can weigth the sum

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/leolr-int/micromamba/envs/py312-poetry/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/home/leolr-int/micromamba/envs/py312-poetry/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 299, in dispatch_control
    await self.process_control(msg)
  File "/home/leolr-int/micromamba/envs/py312-poetry/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 305, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/leolr-int/micromamba/envs/py312-poetry/lib/python3.12/site-packages/jupyter_client/session.py", line 994, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/le